# Step 6 — Hybrid Recommender & Demo Simulation
Loads all artifacts, runs the HybridRecommender, and deploys a SageMaker endpoint.

In [ ]:
import pandas as pd, numpy as np, json, boto3, sagemaker, os

BUCKET = 'hybrid-rec-demo-YOUR_ACCOUNT_ID'   # <-- REPLACE THIS
s3     = boto3.client('s3')
role   = sagemaker.get_execution_role()
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)


In [ ]:
# Download all required artifacts from S3
files_to_download = [
    ('models/mba_rules.json',              '../models/mba_rules.json'),
    ('data/processed/user_features.csv',   '../data/processed/user_features.csv'),
    ('data/processed/products_enriched.csv','../data/processed/products_enriched.csv'),
]
for s3_key, local_path in files_to_download:
    s3.download_file(BUCKET, s3_key, local_path)
    print(f'Downloaded: {local_path}')

# user_clusters.csv comes from the KMeans training job output
# If you ran notebook 04, it was extracted to ../models/kmeans_artifacts/user_clusters.csv
# Alternatively: s3.download_file(BUCKET, 'models/user_clusters.csv', '../models/user_clusters.csv')


In [ ]:
class HybridRecommender:
    """
    Hybrid recommendation engine.
    Score = 0.40*MBA + 0.25*ClusterPop + 0.20*Rating + 0.15*PriceMatch
    """

    def __init__(self, mba_rules_path, user_clusters_path, user_features_path, products_path):
        with open(mba_rules_path) as f:
            self.mba_rules = json.load(f)
        self.user_clusters = pd.read_csv(user_clusters_path)
        self.user_features = pd.read_csv(user_features_path)
        self.products      = pd.read_csv(products_path)
        self._build_cluster_popularity()
        print('HybridRecommender ready.')

    def _build_cluster_popularity(self):
        self.cluster_popular = {}
        for c in self.user_clusters['cluster'].unique():
            self.cluster_popular[c] = self.products.sample(10)['product_id'].tolist()

    def _normalize(self, values):
        arr = np.array(values, dtype=float)
        if arr.max() == arr.min():
            return np.ones_like(arr)
        return (arr - arr.min()) / (arr.max() - arr.min())

    def recommend(self, user_id, viewed_product_id, top_n=5):
        user_row = self.user_features[self.user_features['user_id'] == user_id]
        if user_row.empty:
            return self._fallback_popular(top_n)

        user_region = user_row.iloc[0]['region']
        price_sens  = user_row.iloc[0]['price_sensitivity']

        cluster_row  = self.user_clusters[self.user_clusters['user_id'] == user_id]
        user_cluster = int(cluster_row.iloc[0]['cluster']) if not cluster_row.empty else 0
        user_persona = cluster_row.iloc[0]['persona']     if not cluster_row.empty else 'Unknown'

        mba_candidates = self.mba_rules.get(viewed_product_id, [])
        if not mba_candidates:
            mba_candidates = [{'product': p, 'score': 1.0, 'confidence': 0.5, 'lift': 1.0}
                               for p in self.cluster_popular.get(user_cluster, [])]

        max_price = self.products['price'].max()
        avg_spend = user_row.iloc[0]['avg_order_value']
        filtered  = []

        for cand in mba_candidates:
            prod_row = self.products[self.products['product_id'] == cand['product']]
            if prod_row.empty:
                continue
            prod = prod_row.iloc[0]
            if prod['price'] > avg_spend * 3:
                continue
            norm_price  = prod['price'] / max_price
            price_match = 1 - abs(price_sens - norm_price)
            filtered.append({**cand, 'product_name': prod['name'],
                             'category': prod['category'], 'price': prod['price'],
                             'avg_rating': prod['avg_rating'], 'price_match': price_match})

        if not filtered:
            return self._fallback_popular(top_n)

        cluster_pop_list = self.cluster_popular.get(user_cluster, [])
        for c in filtered:
            c['cluster_pop'] = 1.0 if c['product'] in cluster_pop_list else 0.5

        mba_norm    = self._normalize([c['score']      for c in filtered])
        rating_norm = self._normalize([c['avg_rating'] for c in filtered])
        price_arr   = np.array([c['price_match']  for c in filtered])
        cluster_arr = np.array([c['cluster_pop']  for c in filtered])

        final_scores = 0.40*mba_norm + 0.25*cluster_arr + 0.20*rating_norm + 0.15*price_arr

        for i, c in enumerate(filtered):
            c['final_score'] = round(float(final_scores[i]), 4)

        results = sorted(filtered, key=lambda x: x['final_score'], reverse=True)

        return {
            'user_id': user_id, 'persona': user_persona,
            'region': user_region, 'cluster': user_cluster,
            'viewed_product': viewed_product_id,
            'recommendations': [{
                'product_id': r['product'], 'product_name': r['product_name'],
                'category': r['category'],  'price': r['price'],
                'avg_rating': r['avg_rating'], 'final_score': r['final_score'],
                'score_breakdown': {
                    'mba': round(float(mba_norm[i]), 4),
                    'cluster_pop': r['cluster_pop'],
                    'rating': round(float(rating_norm[i]), 4),
                    'price_match': round(r['price_match'], 4)
                }
            } for i, r in enumerate(results[:top_n])]
        }

    def _fallback_popular(self, top_n):
        return self.products.nlargest(top_n, 'avg_rating')[['product_id','name','price','avg_rating']].to_dict('records')


In [ ]:
# Initialize the recommender
rec = HybridRecommender(
    mba_rules_path     = '../models/mba_rules.json',
    user_clusters_path = '../models/kmeans_artifacts/user_clusters.csv',
    user_features_path = '../data/processed/user_features.csv',
    products_path      = '../data/processed/products_enriched.csv'
)


In [ ]:
# SCENARIO 1 — Champion user viewing Electronics
print('='*60)
print('SCENARIO 1: Champion User viewing Electronics')
print('='*60)
result = rec.recommend(user_id='U0001', viewed_product_id='P001', top_n=5)
print(f"User: {result['user_id']} | Persona: {result['persona']} | Cluster: {result['cluster']}")
print(f"Region: {result['region']} | Viewed: {result['viewed_product']}")
print('\nTop Recommendations:')
for i, r in enumerate(result['recommendations'], 1):
    s = r['score_breakdown']
    print(f"  {i}. {r['product_name']} | ${r['price']:.2f} | ⭐{r['avg_rating']} | Score: {r['final_score']}")
    print(f"     MBA={s['mba']:.2f} | Cluster={s['cluster_pop']:.2f} | Rating={s['rating']:.2f} | Price={s['price_match']:.2f}")


In [ ]:
# SCENARIO 2 — Budget user viewing Books
print('='*60)
print('SCENARIO 2: Budget User viewing Books')
print('='*60)
result2 = rec.recommend(user_id='U0050', viewed_product_id='P010', top_n=5)
print(f"User: {result2['user_id']} | Persona: {result2['persona']} | Cluster: {result2['cluster']}")
for i, r in enumerate(result2['recommendations'], 1):
    print(f"  {i}. {r['product_name']} | ${r['price']:.2f} | Score: {r['final_score']}")


In [ ]:
# SCENARIO 3 — Cold-start (unknown user)
print('='*60)
print('SCENARIO 3: Cold-Start (unknown user)')
print('='*60)
result3 = rec.recommend(user_id='U9999', viewed_product_id='P020', top_n=5)
print('Falling back to globally popular products:')
for i, r in enumerate(result3, 1):
    print(f"  {i}. {r.get('product_id','?')} - {r.get('name',r.get('product_name','?'))} | ${r.get('price',0):.2f}")


In [ ]:
# OPTIONAL: Deploy SageMaker Endpoint
# WARNING: Endpoints are NOT free tier. Delete immediately after testing!

from sagemaker.sklearn.model import SKLearnModel

# Replace with your actual model_data URI from notebook 04
MODEL_DATA_URI = 's3://YOUR_BUCKET/kmeans-training-job-name/output/model.tar.gz'

sklearn_model = SKLearnModel(
    model_data=MODEL_DATA_URI,
    role=role,
    entry_point='inference.py',
    source_dir='../scripts/',
    framework_version='1.0-1'
)

predictor = sklearn_model.deploy(
    initial_instance_count=1,
    instance_type='ml.t2.medium',
    endpoint_name='hybrid-rec-endpoint'
)

print('Endpoint deployed:', predictor.endpoint_name)


In [ ]:
# Test the endpoint
import json
test_payload = {'features': [1200.0, 15, 80.0, 12, 4, 0.10]}
response = predictor.predict(json.dumps(test_payload),
                              initial_args={'ContentType': 'application/json'})
print('Endpoint response:', response)


In [ ]:
# IMPORTANT: Delete endpoint to stop charges
# predictor.delete_endpoint()
# print('Endpoint deleted.')
